In [16]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [33]:
## Load the trained model and preprocessing objects from the project directory
from pathlib import Path

project_dir = Path(r'C:\Users\akank\OneDrive\Desktop\annclassification')
model = load_model(project_dir / 'model.h5')

with open(project_dir / 'onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open(project_dir / 'label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open(project_dir / 'scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [18]:
##exampl input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',  
    'Gender': 'Male',
    'Age': 30,
    'Tenure': 5,
    'Balance': 50000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 60000
}

In [19]:
# Use the encoders saved during training; do not fit a new encoder on one record.


In [20]:
geo_input = pd.DataFrame({'Geography': [input_data['Geography']]})
geo_encoded = onehot_encoder_geo.transform(geo_input).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [21]:
input_data = pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,30,5,50000,2,1,1,60000


In [22]:
##COMBINE ONE-HOT ENCODED COLUMNS WITH INPUT DATA
input_data = pd.concat([input_data.reset_index(drop=True),geo_encoded_df], axis=1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,30,5,50000,2,1,1,60000,1.0,0.0,0.0


In [23]:
##encode  categorical features
input_data['Gender'] = label_encoder_gender.transform(input_data['Gender'])
input_data = input_data.drop(columns=['Geography'])
input_scaled = scaler.transform(input_data)
prediction_probability = model.predict(input_scaled, verbose=0)[0][0]
prediction = int(prediction_probability >= 0.5)
print(f'Churn probability: {prediction_probability:.3f}')
print('Prediction:', 'Exited' if prediction else 'Stayed')

Churn probability: 0.105
Prediction: Stayed


In [26]:
## Final preprocessing (safe to rerun)
input_df = input_data.copy()

if 'Gender' in input_df.columns and input_df['Gender'].dtype.kind in ('O', 'U', 'S'):
    input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

if 'Geography' in input_df.columns:
    input_df = input_df.drop(columns=['Geography'])

for column in geo_encoded_df.columns:
    if column not in input_df.columns:
        input_df[column] = geo_encoded_df[column].values

feature_order = [
    'CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
    'Geography_France', 'Geography_Germany', 'Geography_Spain'
]
input_df = input_df[feature_order]
input_scaled = scaler.transform(input_df)
prediction_probability = model.predict(input_scaled, verbose=0)[0][0]
prediction = int(prediction_probability >= 0.5)
print(f'Churn probability: {prediction_probability:.3f}')
print('Prediction:', 'Exited' if prediction else 'Stayed')

Churn probability: 0.105
Prediction: Stayed


In [27]:
scaler.transform=scaler.transform(input_df)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,30,5,50000,2,1,1,60000,1.0,0.0,0.0


In [28]:
input_scaled = scaler.transform(input_df)
input_scaled

TypeError: 'numpy.ndarray' object is not callable

In [29]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


array([[0.10534564]], dtype=float32)

In [30]:
prediction_proba = prediction[0][0]

In [31]:
prediction_proba

np.float32(0.105345644)

In [32]:
if prediction_proba >= 0.5:
    print('the customer is likely to exit')
else:
    print('the customer is likely to stay')

the customer is likely to stay
